# 07 Build final analytical datasets

This notebook builds the final analytical datasets used across:
- search-demand analysis
- territorial mobility intelligence
- exploratory analysis
- dashboard development
- future modeling workflows

The notebook standardizes:
- tourism demand signals
- territorial outbound mobility structures
- destination metadata
- strategic tourism segmentation variables
- analytical subsets

The objective is to create reusable and consistent analytical datasets.


## 1. Imports

In [40]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path


## 2. Paths

In [41]:
INTERIM_PATH = Path("../data/interim")
PROCESSED_PATH = Path("../data/processed")
RAW_PATH = Path("../data/raw")


## 3. Load datasets

In [42]:
df_ine = pd.read_parquet(
    INTERIM_PATH /
    "01_ine_clean_v2.parquet"
)

df_trends = pd.read_parquet(
    PROCESSED_PATH /
    "06_google_trends_master_v2.parquet"
)

df_monthly_searches = pd.read_parquet(
    INTERIM_PATH /
    "05_dataforseo_monthly_v2.parquet"
)

df_mapping = pd.read_parquet(
    PROCESSED_PATH /
    "03_keyword_mapping_master_v2.parquet"
)


print(df_ine.shape)
print(df_trends.shape)
print(df_monthly_searches.shape)


(1531599, 13)
(21546, 3)
(15769, 6)


---
# PART A - Territorial Mobility Datasets

## 4. Cleaning and normalization

In [43]:
def clean_text(text):
    """
    Normalizes text for cross-dataset merging:
    lowercase, strip whitespace, remove accents.
    """
    if pd.isna(text):
        return np.nan
    text = str(text).lower().strip()
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("utf-8")
    return text

In [44]:
df_mapping = df_mapping.drop(columns=["url", "keyword_type", "source", "raw_title"])
df_mapping.columns

Index(['search_term', 'parent_country'], dtype='object')

### 4.1 Normalize search terms

In [45]:
df_ine["destination_clean"] = (
    df_ine["destination"]
    .apply(clean_text)
)

df_mapping["search_term"] = (
    df_mapping["search_term"]
    .apply(clean_text)
)

df_mapping["parent_country"] = (
    df_mapping["parent_country"]
    .apply(clean_text)
)

In [46]:

df_trends["search_term"] = (
    df_trends["search_term"]
    .apply(clean_text)
)

df_monthly_searches["search_term"] = (
    df_monthly_searches["search_term"]
    .str.replace(
        "viaje a ",
        "",
        regex=False
    )
    .str.strip()
    .apply(clean_text)
)

### 4.2 Normalize period column

In [47]:
for df in [df_ine, df_trends, df_monthly_searches]:
    df["period"] = pd.to_datetime(
        df["period"]
    )

### 4.3 Normalize province_code

In [48]:
df_ine["depart_province_code"] = (
    df_ine["depart_province_code"]
    .apply(clean_text)
)

## 5. Aggregation and signal construction

This section creates:
- national tourism demand structures
- aggregated search signals
- destination-level analytical datasets


### 5.1 Build Google Trends aggregated signal

In [49]:
df_trends = df_trends.merge(
    df_mapping
    .drop_duplicates(),
    on="search_term",
    how="left"
)

df_trends_agg = (
    df_trends
    .groupby(
        ["period", "parent_country"],
        as_index=False
    )
    .agg({
        "trend_index": "mean"
    })
    .rename(
        columns={
            "parent_country": "country"
        }
    )
)

display(df_trends_agg.head())


,period,country,trend_index
0,2018-01-01,albania,0.0
1,2018-01-01,alemania,14.0
2,2018-01-01,andorra,18.0
3,2018-01-01,angola,0.0
4,2018-01-01,arabia saudi,0.0


### 5.2 Build monthly search volume aggregated signal

In [50]:
df_monthly_searches = df_monthly_searches.merge(
    df_mapping,
    on="search_term",
    how="left"
)

df_monthly_searches_agg = (
    df_monthly_searches
    .groupby(
        ["period", "parent_country"],
        as_index=False
    )
    .agg({
        "monthly_searches": "sum",
        "competition": "mean",
        "cpc": "mean"
    })
    .rename(
        columns={
            "parent_country": "country"
        }
    )
)

display(df_monthly_searches_agg.head())


,period,country,monthly_searches,competition,cpc
0,2018-02-01,bahamas,1300,0.60,0.68
1,2018-02-01,brasil,1300,0.66,0.83
2,2018-02-01,chipre,720,0.38,0.48
3,2018-02-01,groenlandia,880,0.42,1.59
4,2018-02-01,guyana,10,0.18,0.36


In [51]:
print("Trend_index (previous and aggregated df):")
print(df_trends.shape)
print(df_trends_agg.shape)
print("Monthly_searches (previous and aggregated df):")
print(df_monthly_searches.shape)
print(df_monthly_searches_agg.shape)

Trend_index (previous and aggregated df):
(23124, 4)
(16287, 3)
Monthly_searches (previous and aggregated df):
(16539, 7)
(13055, 5)


## 6. Feature engineering

This section creates reusable analytical features used for:
- business segmentation
- travel behavior analysis
- long haul vs short haul categorization
- agency-oriented demand analysis
- future feature engineering and modeling

Among some of the enriched data, there are: 
- continent
- EU membership
- Mediterranean destination
- flight dependency
- agency profile
- destination segment
- season
- autonomous_community


### 6.1 Import destination features

In [52]:
df_destination_metadata = pd.read_excel(PROCESSED_PATH / "destination_metadata_enriched.xlsx")

df_destination_metadata["destination_clean"] = (df_destination_metadata["destination"].apply(clean_text))

metadata_columns = [
    "destination_clean",
    "continent",
    "eu",
    "mediterranean",
    "flight_dependency",
    "agency_profile",
    "destination_segment",
    "include_in_analysis"
]


dim_geo = pd.read_excel(
    RAW_PATH / "dim_geography.xlsx"
)

### 6.2 Season classification

In [53]:
def assign_season(month):
    # Exact Easter dates vary yearly but fall within this range
    if month in [6, 7, 8]:
        return "summer"
    elif month == 12:
        return "december"
    elif month in [3,4]:
        return "easter"
    else:
        return "other" 

### 6.3 COVID period classification

In [54]:
def classify_covid_period(date):

    if date < pd.Timestamp("2020-03-01"):
        return "pre_covid"

    elif date <= pd.Timestamp("2021-12-31"):
        return "covid"

    else:
        return "post_covid"


### 6.4 Autonomous community classification

In [55]:
dim_geo = dim_geo.rename(
    columns={
        "CPRO": "depart_province_code",
        "Comunidad Autónoma": "autonomous_community"
    }
)

dim_geo["depart_province_code"] = (
    dim_geo["depart_province_code"]
    .astype(str)
)

## 7. Build master dataset

In [56]:
df_ine_country = df_ine[
    df_ine["destination_type"] == "country"
].copy()

df_master = df_ine_country.merge(
    df_destination_metadata[metadata_columns],
    on="destination_clean",
    how="left"
).copy()

df_master["season"] = (
    df_master["month"]
    .apply(assign_season)
)

df_master["covid_period"] = (
    df_master["period"]
    .apply(classify_covid_period)
)

df_master = df_master.merge(
    dim_geo[
        [
            "depart_province_code",
            "autonomous_community"
        ]
    ],
    on="depart_province_code",
    how="left"
)

df_master_demand_mobility = (
    df_master
    .groupby(
        [
            "period",
            "season",
            "month",
            "year",
            "destination",
            "destination_clean",
            "continent",
            "eu",
            "mediterranean",
            "flight_dependency",
            "agency_profile",
            "destination_segment",
            "include_in_analysis",
            "covid_period"
        ],
        as_index=False
    )["total_tourists"]
    .sum()
)

In [57]:
df_master.shape

(711760, 24)

## 8. Build mobility-only datasets

This section creates territorial outbound mobility datasets for municipality and province-level analysis.

Search-demand variables are intentionally excluded because search signals are only available at aggregated destination-country level.

Replicating national search signals across municipalities would artificially inflate the territorial datasets and create false analytical granularity.


In [58]:
columns_to_drop = [
    "source_file",
    "sheet",
    "destination_type"
]

df_master_mobility = (
    df_master
    .drop(columns=columns_to_drop)
    .copy()
)


### 8.1 Catalonia mobility subset

In [59]:
CATALONIA_PROVINCES = [
    "Barcelona",
    "Girona",
    "Lleida",
    "Tarragona"
]

df_catalonia_mobility = (
    df_master_mobility[
        df_master_mobility[
            "depart_province"
        ].isin(CATALONIA_PROVINCES)
    ]
    .copy()
)


## 9. Territorial subsets

This section creates territorial analytical subsets designed to isolate structurally relevant outbound tourism behavior.

These subsets reduce:
- short-haul distortion
- border effects
- low-interest destinations
- low travel agency dependency


In [60]:
df_mobility_non_europe = (
    df_master_mobility[
        df_master_mobility[
            "include_in_analysis"
        ] == True
    ]
    .copy()
)

df_mobility_commercial_destinations = (
    df_mobility_non_europe[
        df_mobility_non_europe[
            "agency_profile"
        ].isin(["medium", "high"])
    ]
    .copy()
)

df_mobility_commercial_non_usa_eu = (
    df_mobility_commercial_destinations[
        df_mobility_commercial_destinations[
            "destination_clean"
        ] != "estados unidos de america"
    ]
    .copy()
)


## 9.1 Provincial mobility dataset

In [61]:
df_provincial_demand = (
    df_master
    .groupby([
        "period", "season", "month", "year",
        "destination", "destination_clean",
        "depart_province", "autonomous_community",
        "continent", "eu", "mediterranean",
        "flight_dependency", "agency_profile",
        "destination_segment", "include_in_analysis",
        "covid_period"
    ], as_index=False)["total_tourists"]
    .sum()
)

print(f"Shape: {df_provincial_demand.shape}")
print(f"Provinces: {df_provincial_demand['depart_province'].nunique()}")
print(f"Destinations: {df_provincial_demand['destination_clean'].nunique()}")

Shape: (96987, 17)
Provinces: 52
Destinations: 155


--- 
# PART B - National Search-Demand Datasets

## 10. Build national search-demand datasets

This section creates national-level tourism demand datasets aligned with external search signals.

Final dataset granularity:
- period × destination at national level (same granularity as digital demand signals)


In [62]:
df_national_level = (
    df_master
    .groupby(
        [
            "period",
            "season",
            "month",
            "year",
            "destination",
            "destination_clean",
            "continent",
            "eu",
            "mediterranean",
            "flight_dependency",
            "agency_profile",
            "destination_segment",
            "include_in_analysis",
            "covid_period"
        ],
        as_index=False
    )["total_tourists"]
    .sum()
)

In [63]:
df_demand = df_national_level.merge(
    df_trends_agg,
    left_on=[
        "period",
        "destination_clean"
    ],
    right_on=[
        "period",
        "country"
    ],
    how="left"
)

df_demand = df_demand.drop(
    columns=["country"]
).copy()


In [64]:
df_master_demand = df_demand.merge(
    df_monthly_searches_agg,
    left_on=[
        "period",
        "destination_clean"
    ],
    right_on=[
        "period",
        "country"
    ],
    how="left"
)

df_master_demand = df_master_demand.drop(
    columns=["country"]
).copy()


## 11. Search-demand subsets

This section creates strategic subsets optimized for:
- long-haul tourism analysis
- travel agency dependency analysis
- premium tourism behavior
- destination recovery analysis


In [65]:
df_demand_non_europe = (
    df_master_demand[
        df_master_demand[
            "include_in_analysis"
        ] == True
    ]
    .copy()
)

df_demand_commercial_destinations = (
    df_demand_non_europe[
        df_demand_non_europe[
            "agency_profile"
        ].isin(["medium", "high"])
    ]
    .copy()
)

df_demand_commercial_non_usa_eu = (
    df_demand_commercial_destinations[
        df_demand_commercial_destinations[
            "destination_clean"
        ] != "estados unidos de america"
    ]
    .copy()
)


--- 
# Validation & Export

## 12. Quality assurance validation

This section validates:
- dataset granularity
- duplicated observations
- missing values
- metadata consistency
- temporal coherence


In [66]:
datasets = {
    "df_master_mobility": df_master_mobility,
    "df_provincial_demand":  df_provincial_demand,
    "df_catalonia_mobility": df_catalonia_mobility,
    "df_mobility_non_europe": df_mobility_non_europe,
    "df_mobility_commercial_destinations": df_mobility_commercial_destinations,
    "df_mobility_commercial_non_usa_eu": df_mobility_commercial_non_usa_eu,
    "df_master_demand": df_master_demand,
    "df_demand_non_europe": df_demand_non_europe,
    "df_demand_commercial_destinations": df_demand_commercial_destinations,
    "df_demand_commercial_non_usa_eu": df_demand_commercial_non_usa_eu, 
    "df_monthly_searches_agg": df_monthly_searches_agg,
    "df_trends_agg": df_trends_agg
}

for name, df in datasets.items():
    print("=" * 50)
    print(name)
    print(df.shape)
    print(f"Rows with null values: {df.isna().any(axis=1).sum()}")


df_master_mobility
(711760, 21)
Rows with null values: 7
df_provincial_demand
(96987, 17)
Rows with null values: 0
df_catalonia_mobility
(166024, 21)
Rows with null values: 0
df_mobility_non_europe
(74407, 21)
Rows with null values: 0
df_mobility_commercial_destinations
(66815, 21)
Rows with null values: 0
df_mobility_commercial_non_usa_eu
(47049, 21)
Rows with null values: 0
df_master_demand
(8044, 19)
Rows with null values: 384
df_demand_non_europe
(5157, 19)
Rows with null values: 333
df_demand_commercial_destinations
(2945, 19)
Rows with null values: 60
df_demand_commercial_non_usa_eu
(2864, 19)
Rows with null values: 60
df_monthly_searches_agg
(13055, 5)
Rows with null values: 205
df_trends_agg
(16287, 3)
Rows with null values: 0


In [67]:
for name, df in datasets.items():

    print("-" * 50)
    print(name)

    dest_col = "destination" if "destination" in df.columns else "country"

    print(f"Destinations: {df[dest_col].nunique()}")

--------------------------------------------------
df_master_mobility
Destinations: 155
--------------------------------------------------
df_provincial_demand
Destinations: 155
--------------------------------------------------
df_catalonia_mobility
Destinations: 145
--------------------------------------------------
df_mobility_non_europe
Destinations: 114
--------------------------------------------------
df_mobility_commercial_destinations
Destinations: 46
--------------------------------------------------
df_mobility_commercial_non_usa_eu
Destinations: 45
--------------------------------------------------
df_master_demand
Destinations: 155
--------------------------------------------------
df_demand_non_europe
Destinations: 114
--------------------------------------------------
df_demand_commercial_destinations
Destinations: 46
--------------------------------------------------
df_demand_commercial_non_usa_eu
Destinations: 45
--------------------------------------------------
df_m

In [68]:
for name, df in datasets.items():

    print("-" * 50)
    print(name)

    print(
        "Min:",
        df["period"].min()
    )

    print(
        "Max:",
        df["period"].max()
    )

--------------------------------------------------
df_master_mobility
Min: 2019-07-01 00:00:00
Max: 2026-03-01 00:00:00
--------------------------------------------------
df_provincial_demand
Min: 2019-07-01 00:00:00
Max: 2026-03-01 00:00:00
--------------------------------------------------
df_catalonia_mobility
Min: 2019-07-01 00:00:00
Max: 2026-03-01 00:00:00
--------------------------------------------------
df_mobility_non_europe
Min: 2019-07-01 00:00:00
Max: 2026-03-01 00:00:00
--------------------------------------------------
df_mobility_commercial_destinations
Min: 2019-07-01 00:00:00
Max: 2026-03-01 00:00:00
--------------------------------------------------
df_mobility_commercial_non_usa_eu
Min: 2019-07-01 00:00:00
Max: 2026-03-01 00:00:00
--------------------------------------------------
df_master_demand
Min: 2019-07-01 00:00:00
Max: 2026-03-01 00:00:00
--------------------------------------------------
df_demand_non_europe
Min: 2019-07-01 00:00:00
Max: 2026-03-01 00:00:00

In [69]:
for name, df in datasets.items():

    print("=" * 50)
    print(name)
    
    if "monthly_searches" in df.columns:
        dest_col = "destination" if "destination" in df.columns else "country"
        missing = (
            df[df["monthly_searches"].isna()][dest_col]
            .unique()
        )

        print(missing)

    else:

        print(
            "No monthly_searches column"
        )

df_master_mobility
No monthly_searches column
df_provincial_demand
No monthly_searches column
df_catalonia_mobility
No monthly_searches column
df_mobility_non_europe
No monthly_searches column
df_mobility_commercial_destinations
No monthly_searches column
df_mobility_commercial_non_usa_eu
No monthly_searches column
df_master_demand
['Arabia Saudí' 'Belarús' 'Camerún' 'Iraq' 'Irán' 'Kazajstán'
 'Otros países de Asia' 'Pakistán' 'Santo Tomé y Príncipe' 'Azerbaiyán'
 'Mali' 'Sierra Leona' 'Siria' 'Swazilandia' 'Tayikistán' 'Turkmenistán'
 'Benin' 'Bangladesh' 'Burkina Faso' 'Chad' 'Otros países de Europa'
 'Bahréin' 'Guinea' 'Djibouti' 'Níger' 'República Democrática del Congo'
 'Togo']
df_demand_non_europe
['Arabia Saudí' 'Camerún' 'Iraq' 'Irán' 'Kazajstán' 'Otros países de Asia'
 'Pakistán' 'Santo Tomé y Príncipe' 'Azerbaiyán' 'Mali' 'Sierra Leona'
 'Siria' 'Swazilandia' 'Tayikistán' 'Turkmenistán' 'Benin' 'Bangladesh'
 'Burkina Faso' 'Chad' 'Bahréin' 'Guinea' 'Djibouti' 'Níger'
 'Repúbl

In [70]:
# columns with nulls at df_master_demand
print(df_master_demand.isnull().sum()[df_master_demand.isnull().sum() > 0])

# Destinations without trend_index
no_trends = df_master_demand[df_master_demand["trend_index"].isna()]["destination_clean"].unique()
print(f"Destinacions sense trend_index: {len(no_trends)}")
print(sorted(no_trends))

# Destinations without monthly_searches
no_searches = df_master_demand[df_master_demand["monthly_searches"].isna()]["destination_clean"].unique()
print(f"\nDestinacions sense monthly_searches: {len(no_searches)}")
print(sorted(no_searches))

trend_index          62
monthly_searches    325
competition         325
cpc                 325
dtype: int64
Destinacions sense trend_index: 22
['australia', 'egipto', 'guatemala', 'guinea ecuatorial', 'honduras', 'hungria', 'moldavia', 'oman', 'otros paises de asia', 'republica checa', 'senegal', 'singapur', 'sudafrica', 'suiza', 'tailandia', 'tanzania', 'tunez', 'turquia', 'ucrania', 'uganda', 'uruguay', 'venezuela']

Destinacions sense monthly_searches: 27
['arabia saudi', 'azerbaiyan', 'bahrein', 'bangladesh', 'belarus', 'benin', 'burkina faso', 'camerun', 'chad', 'djibouti', 'guinea', 'iran', 'iraq', 'kazajstan', 'mali', 'niger', 'otros paises de asia', 'otros paises de europa', 'pakistan', 'republica democratica del congo', 'santo tome y principe', 'sierra leona', 'siria', 'swazilandia', 'tayikistan', 'togo', 'turkmenistan']


### Search Signal Coverage

Not all destinations have matching digital demand signals:

- **22 destinations** have no `trend_index` — Google Trends returned zero values consistently, likely due to low search volume relative 
  to the anchor keyword benchmark.
- **27 destinations** have no `monthly_searches` — these countries were not included in the keyword mapping pipeline (low commercial 
  relevance or absent from scraped sources).
- **`otros paises de asia/europa`** — INE residual categories with no keyword correspondence. Expected and unresolvable.

These null values are retained in the dataset. Downstream analysis filters to destinations with valid signal coverage where required.

## 13. Export final datasets

This section exports the final analytical datasets used across:
- exploratory analysis
- territorial intelligence
- search-demand analysis
- dashboard development
- future modeling workflows


In [71]:
for name, df in datasets.items():

    parquet_name = (
        "07_"
        + name.replace("df_", "")
        + ".parquet"
    )

    df.to_parquet(
        PROCESSED_PATH /
        parquet_name
    )

    print(
        f"Exported: {parquet_name}"
    )

Exported: 07_master_mobility.parquet
Exported: 07_provincial_demand.parquet
Exported: 07_catalonia_mobility.parquet
Exported: 07_mobility_non_europe.parquet
Exported: 07_mobility_commercial_destinations.parquet
Exported: 07_mobility_commercial_non_usa_eu.parquet
Exported: 07_master_demand.parquet
Exported: 07_demand_non_europe.parquet
Exported: 07_demand_commercial_destinations.parquet
Exported: 07_demand_commercial_non_usa_eu.parquet
Exported: 07_monthly_searches_agg.parquet
Exported: 07_trends_agg.parquet


In [72]:
df_master_demand.to_excel(PROCESSED_PATH / '07_df_master_demand.xlsx')

**Note**: Morocco is only included in "all destinations" dfs to reduce structural noise from VFR (visiting friends and relatives) and migration-related mobility, which would otherwise inflate the signal with non-leisure travel behavior.